# SparkCut — entrenamiento y validación del modelo RNN v6

Este notebook es el artefacto reproducible de entrenamiento de SparkCut. La aplicación Django no usa el antiguo notebook v3: usa los artefactos de `proyecto final/ml_artifacts/` generados por el pipeline v6.

**Objetivo:** ordenar ventanas de texto transcrito para encontrar momentos que sobresalen dentro de un video largo. El puntaje es una señal de relevancia relativa, no una probabilidad de viralidad futura.

La lógica de entrenamiento está versionada en `train_highlight_ranker.py`, `tune_highlight_hybrid.py` y `export_contextual_ranker.py`. Este notebook los ejecuta y valida sus artefactos, evitando duplicar código y reducir el riesgo de que el notebook y Django queden desalineados.


## Contrato de reproducibilidad

1. Ejecutar las celdas en orden, de arriba hacia abajo.
2. `RUN_TRAINING=False` solo inspecciona los artefactos ya entrenados; no descarga el dataset grande.
3. Para reentrenar en Colab, cambiarlo a `True`, usar GPU y conservar la caché de Hugging Face en Drive.
4. La selección del modelo contextual usa validación; el conjunto de prueba oficial se reporta una sola vez al final.
5. La copia que consume Django debe coincidir byte a byte con `entrenamiento/sparkcut_highlights_v6/`.


In [ ]:
# Dependencias (ejecutar una vez en Colab)
%pip install -q datasets sentencepiece scikit-learn scipy pandas joblib torch


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "entrenamiento").exists():
    # Ajusta esta ruta si clonaste el repositorio en otra carpeta de Colab.
    REPO_ROOT = Path("/content/proyecto-de-recortes-spark-cut-")
TRAIN_ROOT = REPO_ROOT / "entrenamiento"
APP_ROOT = REPO_ROOT / "proyecto final"
ARTIFACTS = APP_ROOT / "ml_artifacts"
RUN_TRAINING = False

assert TRAIN_ROOT.exists(), f"No se encontró entrenamiento/: {TRAIN_ROOT}"
assert APP_ROOT.exists(), f"No se encontró proyecto final/: {APP_ROOT}"
print("Repositorio:", REPO_ROOT)
print("Artefactos Django:", ARTIFACTS)


In [ ]:
# Entrenamiento canónico v6 (tarda y requiere descargar CBC si RUN_TRAINING=True)
if RUN_TRAINING:
    for script in ("train_highlight_ranker.py", "tune_highlight_hybrid.py", "export_contextual_ranker.py"):
        print(f"\n>>> Ejecutando {script}")
        subprocess.run([sys.executable, script], cwd=TRAIN_ROOT, check=True)
    print("Entrenamiento y exportación v6 terminados.")
else:
    print("Inspección local: se usan los artefactos ya entrenados. Cambia RUN_TRAINING=True para reentrenar.")


In [ ]:
# Validación estructural de los artefactos que carga Django
required = [
    "config.json", "metrics.json", "golden_tests.json", "tokenizer.model",
    "context_ranker.joblib", "model_0.pt", "model_1.pt", "model_2.pt",
]
missing = [name for name in required if not (ARTIFACTS / name).is_file()]
assert not missing, f"Faltan artefactos: {missing}"
config = json.loads((ARTIFACTS / "config.json").read_text(encoding="utf-8"))
metrics = json.loads((ARTIFACTS / "metrics.json").read_text(encoding="utf-8"))
assert config["version"] == 6, config.get("version")
assert config["n_models"] == 3
assert config["position_independent"] is True
assert metrics["target_is_future_virality"] is False
print("Modelo:", config["version"], "| ensamble:", config["n_models"], "RNN BiGRU")
print("Precision@3:", round(metrics["test_precision_at_3"] * 100, 2), "%")
print("Spearman:", round(metrics["test_spearman_per_video"], 4))


In [ ]:
# Prueba de carga e inferencia con la misma clase usada en entrenamiento
sys.path.insert(0, str(TRAIN_ROOT))
from sparkcut_django.inference import SparkCutScorer

scorer = SparkCutScorer(ARTIFACTS)
texts = [
    "Welcome everyone, today we will review the agenda and the usual setup.",
    "The robot suddenly turned around, hit him on the back of the head, and everyone started laughing.",
    "Remember to subscribe and we will continue after this short break.",
]
base = scorer.score_texts(texts)
rnn_scores = [row["score"] for row in base]
context = scorer.score_contextual_ranks(texts, rnn_scores)
assert len(rnn_scores) == len(texts)
assert len(context["combined"]) == len(texts)
print("Puntajes base:", rnn_scores)
print("Ranking contextual:", [round(v * 100, 1) for v in context["combined"]])


In [ ]:
# Verificación de paridad: entrenamiento v6 y Django deben tener los mismos bytes
TRAIN_ARTIFACTS = TRAIN_ROOT / "sparkcut_highlights_v6"
if TRAIN_ARTIFACTS.exists():
    for name in required:
        a, b = TRAIN_ARTIFACTS / name, ARTIFACTS / name
        if a.is_file() and b.is_file():
            assert hashlib.sha256(a.read_bytes()).digest() == hashlib.sha256(b.read_bytes()).digest(), name
    print("Paridad confirmada entre sparkcut_highlights_v6 y proyecto final/ml_artifacts.")
else:
    print("No se encontró la carpeta de entrenamiento local; se validaron los artefactos de Django.")


## Resultados y lectura correcta

La evaluación registrada usa 545 videos de prueba intactos: Precision@3 = 0.3278 frente a 0.2954 de la línea base RNN y 0.2000 de selección aleatoria; Spearman por video = 0.0626 frente a 0.0502.

Estas métricas miden ordenamiento de momentos destacados dentro del video usando replay como señal indirecta. No demuestran ni prometen viralidad futura. El modelo procesa principalmente texto transcrito; no usa la imagen como señal principal.

## Checklist para la sustentación

- [ ] Ejecutar todas las celdas en orden y guardar el notebook sin errores.
- [ ] Mostrar `config.version = 6` y la paridad de artefactos.
- [ ] Mostrar `metrics.json` y explicar Precision@3, baseline y limitaciones.
- [ ] Probar tres videos reales desde Django con `MOCK_SCORER=0`.
- [ ] Conservar la versión del notebook junto con el commit del repositorio.
